##Notebook 3: Contract Fingerprint

This notebook converts every contract into a structured machine-readable representation called a Contract Fingerprint.

Rather than processing raw legal text throughout the pipeline, MeridianIQ creates a compact representation containing binary clause indicators, entity-level metadata, lifecycle information, and business-specific features. This standardized fingerprint becomes the primary input for clause detection, semantic retrieval, risk scoring, and LLM-powered report generation.

By separating raw contracts from structured intelligence, MeridianIQ builds a reusable representation that allows every downstream component to reason consistently about the same contract.

# Loading Libraries and Processed Data

In [1]:
import pandas as pd
import numpy as np
import json,re
from column_groups import *

pd.set_option("display.max_columns",None)
pd.set_option("display.max_colwidth",200)

ModuleNotFoundError: No module named 'column_groups'

In [ ]:
df=pd.read_csv("master_clauses.csv")
risk_config=pd.read_csv("clause_risk_config.csv")

In [ ]:
df.shape
risk_config.shape

(33, 11)

##Binary Contract Fingerprint

The first step focuses on binary legal clauses.

Each Yes/No clause is converted into a numerical feature indicating whether the clause exists within a contract.

Instead of repeatedly parsing clause answers, every contract now has a standardized binary representation.

In [ ]:
binary_fingerprint=pd.DataFrame()
binary_fingerprint["filename"]=df["Filename"]

for col in binary_cols:
  clean_name=(
      col.replace("-Answer","")
      .replace("- Answer","")
      .strip()
  )

  snake_name=(
      clean_name.lower()
      .replace("/","_")
      .replace("-","_")
      .replace(" ","_")
  )

  binary_fingerprint[snake_name]=(
    df[col]
    .astype(str)
    .str.strip()
    .map({"Yes":True, "No":False})
  )
binary_fingerprint.head()


,filename,most_favored_nation,competitive_restriction_exception,non_compete,exclusivity,no_solicit_of_customers,no_solicit_of_employees,non_disparagement,termination_for_convenience,rofr_rofo_rofn,change_of_control,anti_assignment,revenue_profit_sharing,price_restrictions,minimum_commitment,volume_restriction,ip_ownership_assignment,joint_ip_ownership,license_grant,non_transferable_license,affiliate_license_licensor,affiliate_license_licensee,unlimited_all_you_can_eat_license,irrevocable_or_perpetual_license,source_code_escrow,post_termination_services,audit_rights,uncapped_liability,cap_on_liability,liquidated_damages,warranty_duration,insurance,covenant_not_to_sue,third_party_beneficiary
0,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf,False,False,False,False,False,False,True,False,False,False,True,False,False,True,False,False,False,True,False,False,False,False,False,False,False,True,False,True,False,True,False,False,False
1,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B(01)_525118_EX-10.B(01)_Content License Agreement.pdf,True,False,False,False,False,False,False,True,False,False,True,True,False,True,False,False,False,True,False,False,False,True,False,False,False,True,True,True,False,False,False,False,False
2,FulucaiProductionsLtd_20131223_10-Q_EX-10.9_8368347_EX-10.9_Content License Agreement.pdf,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,True,True,False,False,False,True,False,False,False,False,False,False,False
3,GopageCorp_20140221_10-K_EX-10.1_8432966_EX-10.1_Content License Agreement.pdf,False,False,False,True,False,False,False,False,False,True,True,True,False,False,False,False,False,True,True,False,False,False,False,False,False,True,True,True,False,False,False,False,False
4,IdeanomicsInc_20160330_10-K_EX-10.26_9512211_EX-10.26_Content License Agreement.pdf,False,False,False,False,False,False,False,False,True,False,True,True,False,False,False,False,False,True,False,False,False,False,True,False,False,True,True,True,False,False,False,False,False


##Entity Contract Fingerprint

Not every important contract attribute is binary.

This step extracts those entity-style fields and combines them into the contract fingerprint.

These metadata features improve explainability and later provide valuable context during report generation and contract analysis.

In [ ]:
entity_fingerprint=pd.DataFrame()
entity_fingerprint["filename"]=df["Filename"]

for col in entity_cols:
  clean_name=(
      col.replace("-Answer","")
      .replace("- Answer","")
      .strip()
  )

  snake_name=(
      clean_name.lower()
      .replace("/","_")
      .replace("-","_")
      .replace(" ","_")
  )

  entity_fingerprint[snake_name]=df[col]

entity_fingerprint.head()

,filename,document_name,parties,agreement_date,effective_date,expiration_date,renewal_term,notice_period_to_terminate_renewal,governing_law
0,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf,MARKETING AFFILIATE AGREEMENT,"Birch First Global Investments Inc. (""Company""); Mount Kowledge Holdings Inc. (""Marketing Affiliate"", ""MA"")",5/8/14,NaN,12/31/14,successive 1 year,30 days,Nevada
1,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B(01)_525118_EX-10.B(01)_Content License Agreement.pdf,VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT,"Rogers Cable Communications Inc. (""Rogers""); EuroMedia Holdings Corp. (""Licensor"")",7/11/06,7/11/06,6/30/10,2 years,60 days,"Ontario, Canada"
2,FulucaiProductionsLtd_20131223_10-Q_EX-10.9_8368347_EX-10.9_Content License Agreement.pdf,CONTENT DISTRIBUTION AND LICENSE AGREEMENT,"CONVERGTV, INC. (“ConvergTV”); Fulucai Productions Ltd. (""Producer"")",11/15/12,11/15/12,NaN,"perpetual, 11/15/2014",NaN,Florida
3,GopageCorp_20140221_10-K_EX-10.1_8432966_EX-10.1_Content License Agreement.pdf,WEBSITE CONTENT LICENSE AGREEMENT,"PSiTech Corporation (""Licensor""); Empirical Ventures, Inc (""Licensee"")",2/10/14,2/10/14,2/10/19,3 years,90 days,Nevada
4,IdeanomicsInc_20160330_10-K_EX-10.26_9512211_EX-10.26_Content License Agreement.pdf,CONTENT LICENSE AGREEMENT,"Beijing Sun Seven Stars Culture Development Limited (""Licensor""); YOU ON DEMAND HOLDINGS, INC (""Licensee"")",12/21/15,12/21/15,12/21/35,NaN,NaN,New York


##Building the Complete Contract Fingerprint

The binary fingerprint and entity fingerprint are combined into a single unified representation.

In [ ]:
contract_fingerprints=entity_fingerprint.merge(binary_fingerprint,
                                               on="filename",
                                               how="left")
contract_fingerprints.shape

(510, 42)

In [ ]:
contract_fingerprints.head(2)

,filename,document_name,parties,agreement_date,effective_date,expiration_date,renewal_term,notice_period_to_terminate_renewal,governing_law,most_favored_nation,competitive_restriction_exception,non_compete,exclusivity,no_solicit_of_customers,no_solicit_of_employees,non_disparagement,termination_for_convenience,rofr_rofo_rofn,change_of_control,anti_assignment,revenue_profit_sharing,price_restrictions,minimum_commitment,volume_restriction,ip_ownership_assignment,joint_ip_ownership,license_grant,non_transferable_license,affiliate_license_licensor,affiliate_license_licensee,unlimited_all_you_can_eat_license,irrevocable_or_perpetual_license,source_code_escrow,post_termination_services,audit_rights,uncapped_liability,cap_on_liability,liquidated_damages,warranty_duration,insurance,covenant_not_to_sue,third_party_beneficiary
0,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf,MARKETING AFFILIATE AGREEMENT,"Birch First Global Investments Inc. (""Company""); Mount Kowledge Holdings Inc. (""Marketing Affiliate"", ""MA"")",5/8/14,NaN,12/31/14,successive 1 year,30 days,Nevada,False,False,False,False,False,False,True,False,False,False,True,False,False,True,False,False,False,True,False,False,False,False,False,False,False,True,False,True,False,True,False,False,False
1,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B(01)_525118_EX-10.B(01)_Content License Agreement.pdf,VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT,"Rogers Cable Communications Inc. (""Rogers""); EuroMedia Holdings Corp. (""Licensor"")",7/11/06,7/11/06,6/30/10,2 years,60 days,"Ontario, Canada",True,False,False,False,False,False,False,True,False,False,True,True,False,True,False,False,False,True,False,False,False,True,False,False,False,True,True,True,False,False,False,False,False


##Engineering Lifecycle Features

Raw contract dates contain valuable business information that is difficult to interpret directly.

This step derives additional lifecycle-related features from the available contract metadata.

Examples include contract duration, renewal information, expiration behaviour, and other temporal characteristics that help MeridianIQ better understand how contracts evolve over time.

In [ ]:
contract_fingerprints["has_renewal_term"]=contract_fingerprints["renewal_term"].notna()
contract_fingerprints["has_notice_period_to_terminate_renewal"] =(
    contract_fingerprints["notice_period_to_terminate_renewal"].notna()
)
contract_fingerprints["has_expiration_date"] =contract_fingerprints["expiration_date"].notna()
contract_fingerprints["has_effective_date"] =contract_fingerprints["effective_date"].notna()
contract_fingerprints["has_governing_law"] =contract_fingerprints["governing_law"].notna()

##Constructing the MVP Contract Fingerprint

Although the complete fingerprint contains every available feature, MeridianIQ Version 1 focuses on a carefully selected subset of high-value clauses.

This step filters the complete fingerprint using the MVP clause configuration created in Notebook 02.

Restricting the fingerprint to MVP clauses reduces complexity without sacrificing practical business value.

In [ ]:
mvp_config=risk_config[risk_config["is_mvp_clause"]==True].copy()

mvp_clause_names=mvp_config["clause_name"].to_list()
mvp_snake_cols=[
    name.lower()
    .replace("/","_")
    .replace("-","_")
    .replace(" ","_")
    for name in mvp_clause_names
]

metadata_cols=[
    "filename",
    "document_name",
    "parties",
    "agreement_date",
    "effective_date",
    "expiration_date",
    "renewal_term",
    "notice_period_to_terminate_renewal",
    "governing_law",
    "has_renewal_term",
    "has_notice_period_to_terminate_renewal",
    "has_expiration_date",
    "has_effective_date",
    "has_governing_law"
]

mvp_cols=metadata_cols+mvp_snake_cols
mvp_contract_fingerprints=contract_fingerprints[mvp_cols].copy()
mvp_contract_fingerprints.shape


(510, 37)

##Building the Clause Context Table

Each binary clause indicates whether a clause exists, but it does not preserve the original legal language.

This step creates a dedicated table containing the textual evidence associated with every clause.

In [ ]:
context_records=[]

for _,row in df.iterrows():
  filename=row["Filename"]

  for col in context_cols:
    clean_name=col.strip()
    context_text=row[col]

    if pd.notna(context_text) and str(context_text).strip() not in ["[]","nan",""]:
      context_records.append({
          "filename":filename,
          "clause_name":clean_name,
          "context_text":context_text

      })

clause_contexts=pd.DataFrame(context_records)
clause_contexts.head()


,filename,clause_name,context_text
0,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf,Document Name,['MARKETING AFFILIATE AGREEMENT']
1,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf,Parties,"['BIRCH FIRST GLOBAL INVESTMENTS INC.', 'MA', 'Marketing Affiliate', 'MOUNT KNOWLEDGE HOLDINGS INC.', 'Company']"
2,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf,Agreement Date,"['8th day of May 2014', 'May 8, 2014']"
3,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf,Effective Date,['This agreement shall begin upon the date of its execution by MA and acceptance in writing by Company']
4,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf,Expiration Date,['This agreement shall begin upon the date of its execution by MA and acceptance in writing by Company and shall remain in effect until the end of the current calendar year and shall be automatica...


In [ ]:
clause_contexts.shape

(6702, 3)

##Building the Clause Answer Table

Alongside the clause context table, MeridianIQ stores the structured clause answers in a separate table.

This separation keeps the overall architecture clean and avoids mixing raw text with structured features.

In [ ]:
answer_records=[]

for _,row in df.iterrows():
  filename=row["Filename"]

  for col in answer_cols:
    clean_name=(
        col.replace("-Answer","")
        .replace("- Answer","")
        .strip()
    )

    answer=row[col]
    answer_records.append({
        "filename":filename,
        "clause_name":clean_name,
        "answer":answer
    })

clause_answers=pd.DataFrame(answer_records)
clause_answers.head()

,filename,clause_name,answer
0,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf,Document Name,MARKETING AFFILIATE AGREEMENT
1,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf,Parties,"Birch First Global Investments Inc. (""Company""); Mount Kowledge Holdings Inc. (""Marketing Affiliate"", ""MA"")"
2,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf,Agreement Date,5/8/14
3,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf,Effective Date,NaN
4,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf,Expiration Date,12/31/14


In [ ]:
clause_answers.shape

(20910, 3)

##Validating Against the Risk Configuration

The contract fingerprint should remain consistent with the risk taxonomy created in Notebook 02.

This validation step confirms that every MVP clause defined in the risk configuration is also present within the generated fingerprint.

In [ ]:
config_clauses=set(risk_config["clause_name"])
answer_clauses=set(clause_answers["clause_name"])

missing_in_answers=config_clauses-answer_clauses
missing_in_config=answer_clauses-config_clauses

missing_in_answers,missing_in_config

(set(),
 {'Agreement Date',
  'Document Name',
  'Effective Date',
  'Expiration Date',
  'Governing Law',
  'Notice Period To Terminate Renewal',
  'Parties',
  'Renewal Term'})

In [ ]:
binary_clause_names=[
    col.replace("-Answer","")
    .replace("- Answer","")
    .strip()
    for col in binary_cols
]
set(binary_clause_names)-config_clauses

set()

##Exporting Processed Artifacts

After constructing the complete fingerprint, all engineered datasets are exported for reuse.

In [ ]:
contract_fingerprints.to_csv("contract_fingerprints.csv",index=False)
mvp_contract_fingerprints.to_csv("mvp_contract_fingerprints.csv",index=False)

clause_contexts.to_csv("clause_contexts.csv",index=False)
clause_answers.to_csv("clause_answers.csv",index=False)


In [ ]:
pd.read_csv("contract_fingerprints.csv").shape

(510, 47)

In [ ]:
pd.read_csv("mvp_contract_fingerprints.csv").shape

(510, 37)

##Notebook Summary

In this notebook, MeridianIQ transformed raw contract annotations into a structured Contract Fingerprint capable of representing each commercial agreement in a consistent, machine-readable format.

The notebook produced:

1.Binary contract fingerprints

2.Entity-level contract fingerprints

3.Unified contract fingerprints

4.Lifecycle features

5.MVP contract fingerprints

6.Clause context table

7.Clause answer table

8.Validated processed datasets

Together, these artifacts form MeridianIQ's central contract representation layer and provide the structured foundation required for clause detection, semantic evidence retrieval, business risk scoring, and LLM-powered executive contract intelligence.